# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gulgumusdere/flyrank-internship-ml/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**Lane: CTR / Engagement Opportunity Scoring.**

**Task type: scoring / ranking**, not classification. The decision this supports is "which pages should an editor review first for a title/meta/content fix," and editor capacity is limited (a fixed number of reviews per month, same shape as the refresh lane's 50-slot constraint). A yes/no label doesn't tell us *which* yes/no cases to act on first — we need an ordered priority list, so scoring/ranking is the right frame.


In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

The real target — "would fixing this page's title/meta actually raise its CTR" — isn't measurable without running the fix and waiting. So we predict a **proxy: `ctr_gap`**, the difference between a page's actual CTR and the expected CTR for its position tier (median CTR of pages in the same tier).

```
ctr_gap = expected_ctr(position_tier) - actual_ctr
```

This is an **observed** signal (both `ctr` and `position_tier` are measured facts about the page), not a rule someone defined by hand — it's different from FlyRank's own `needs_ctr_fix` product flag, which we deliberately don't have access to and wouldn't want to just reproduce anyway. A minimum volume filter (`impressions_90d >= 50`) removes noisy low-traffic rows before computing this (a page with 1 impression and 1 click shows 100% CTR, which is pure noise, not a signal).


In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Precision@50**: of the top 50 pages ranked by `ctr_gap` (ties broken by `impressions_90d`, since a bigger gap on a higher-traffic page represents a bigger real opportunity), how many have both a meaningfully positive gap and enough impression volume to matter? This mirrors the refresh lane's 24%-vs-74% story — the same idea of asking "did our limited review slots go to the pages that actually needed them."


In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [4]:
import pandas as pd

df = pd.read_csv('../../data/raw/content_refresh_anonymized.csv')

# Minimum volume filter -- removes low-impression noise
# (e.g. impressions=1, clicks=1 shows as 100% CTR, which is not a real signal)
sub = df[df['impressions_90d'] >= 50].copy()
print(f"Rows before filter: {len(df)} / after filter: {len(sub)}")

# Expected CTR per position tier (median -- robust to outliers)
expected = sub.groupby('position_tier')['ctr'].median()
print("\nExpected (median) CTR by position tier:")
print(expected)

sub['expected_ctr'] = sub['position_tier'].map(expected)
sub['ctr_gap'] = sub['expected_ctr'] - sub['ctr']

# Rank: biggest gap first, tie-broken by impressions (bigger audience = bigger opportunity)
ranked = sub.sort_values(['ctr_gap', 'impressions_90d'], ascending=[False, False])

print("\nUnit of analysis -- one row = one content/page:")
ranked[['content_id', 'position_tier', 'impressions_90d', 'ctr', 'expected_ctr', 'ctr_gap']].head(10)


Rows before filter: 30000 / after filter: 23521

Expected (median) CTR by position tier:
position_tier
deep        0.00
page_1      0.22
page_3_5    0.05
striking    0.14
top_3       0.18
Name: ctr, dtype: float64

Unit of analysis -- one row = one content/page:


,content_id,position_tier,impressions_90d,ctr,expected_ctr,ctr_gap
7445,content_c8e9d6ab9013,page_1,208678,0.0,0.22,0.22
23220,content_f986bd514b6e,page_1,22456,0.0,0.22,0.22
25462,content_825a9788af8d,page_1,16786,0.0,0.22,0.22
9443,content_8ba781dafa55,page_1,16156,0.0,0.22,0.22
12869,content_5d5653c4eb4f,page_1,15101,0.0,0.22,0.22
28588,content_847a841969a2,page_1,14519,0.0,0.22,0.22
17362,content_c82bc0c24241,page_1,13676,0.0,0.22,0.22
94,content_9983d31c53cb,page_1,7737,0.0,0.22,0.22
2491,content_d3aaf7d5f2fc,page_1,7732,0.0,0.22,0.22
17230,content_5195668f06db,page_1,6635,0.0,0.22,0.22


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A simple hand rule like "flag if CTR < 0.5%" ignores position entirely, and the data shows why that's unsafe: **the `page_1` tier's median CTR (0.22) came out higher than `top_3`'s (0.18)** in this dataset — the intuitive "better position always means higher CTR" assumption doesn't hold cleanly. A single global threshold would misjudge pages depending on which tier they're in. Computing an expected CTR *per tier* and ranking by the gap adjusts for this automatically, which a one-line rule can't do without turning into a long list of manually-tuned per-tier thresholds — at which point we're just doing the tier-conditional analysis by hand, badly.


In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

Note on the target: `ctr_gap` is derived from *observed* signals (`ctr`, `position_tier`), not from a hand-defined rule like FlyRank's own `needs_ctr_fix` flag — which keeps this an honest ML framing rather than an attempt to reverse-engineer an existing product rule.
